In [1]:
#| default_exp frida

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

/usr/local/lib/python3.10/dist-packages/nbdev/export.py:80: UserWarning: Notebook '/workspaces/gpt/rugptxl_converter.ipynb' uses `#|export` without `#|default_exp` cell.
Note nbdev2 no longer supports nbdev1 syntax. Run `nbdev_migrate` to upgrade.
See https://nbdev.fast.ai/getting_started.html for more information.
  warn(f"Notebook '{nbname}' uses `#|export` without `#|default_exp` cell.\n"


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [4]:
#| export
from os import getenv
model_path = getenv("MODEL")

In [5]:
model_path = 'fred'

In [6]:
#| export
from optimum.onnxruntime import ORTModelForSeq2SeqLM

In [7]:
#| export
seq_length = 1024

full_path = f'./models/{model_path}'
import torch
from transformers import GPT2Tokenizer, T5ForConditionalGeneration, AutoTokenizer

In [8]:
def convert_and_save_model(model_path, save_dir):
    model = ORTModelForSeq2SeqLM.from_pretrained(model_path, export=True)
    model.save_pretrained(save_dir)
    
    # Also save the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenizer.save_pretrained(save_dir)
    
    print(f"Model and tokenizer saved to {save_dir}")

In [9]:
#convert_and_save_model(full_path, full_path+'/optimized')

In [10]:
#| export
tokenizer = GPT2Tokenizer.from_pretrained(full_path+'/optimized/', eos_token='</s>')
model = ORTModelForSeq2SeqLM.from_pretrained(full_path+'/optimized/', provider="CUDAExecutionProvider")

2024-09-30 16:10:07.103944368 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2024-09-30 16:10:07.103969937 [W:onnxruntime:, session_state.cc:1168 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2024-09-30 16:10:07.802927687 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2024-09-30 16:10:07.802944177 [W:onnxruntime:, session_state.cc:1168 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2024-09-30 16:10:08.683789331 [W:onnxrun

In [11]:
#| export
def iftoken(tokenizer, tokens):
    # returns token id if the given string is one token
    token_ids = [tokenizer.encode(token, add_special_tokens=False) for token in tokens]
    return [id for sublist in token_ids for id in sublist if len(sublist) == 1]

In [12]:
#| export
from front.common import process_seq

def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool, temperature:float=0.5):
    blocked_tokens = ['[', '(', '\xa0', '*', '­', '~', '_', '\\', '\uf04a', '\ufeff', '\u2028']
    if not allow_linebreak:
        blocked_tokens.extend(['\n', '\n\n',' \n'])
    bad_words_ids = iftoken(tokenizer, blocked_tokens)
    bad_words_ids = [[w] for w in bad_words_ids]
    
    lm_text = '<LM>' + prompt
    input_ids=torch.tensor([tokenizer.encode(lm_text)]).cuda()
    torch.cuda.empty_cache()
    output_ids = model.generate(input_ids, do_sample=True, temperature=temperature, repetition_penalty=5.0, typical_p=0.9, top_k=10, top_p=0.95, #watermark=False,
                        max_new_tokens=length, bad_words_ids = bad_words_ids,
                        num_return_sequences=num_samples,)
    
    result = [tokenizer.decode(o[1:]).replace('\n', ' ') for o in output_ids]
    result = process_seq(result)
    return result


In [13]:
%%time
get_sample('<LM>На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.24 s, sys: 207 ms, total: 1.45 s
Wall time: 975 ms


[' – просто говно. Вот и все».',
 ' – просто говно. И не надо мне говорить, что я сам виноват в своей жизни». Я подумал: «Вот так и у меня с этим делом…» Но вслух сказал другое… Впрочем нет! Не совсем то же самое!..',
 ' – просто дерьмо». Но, как и в случае с «Апокалипсисом», это был не совсем тот случай.',
 ' – говно». И я понял, что не могу ничего с собой поделать. Я даже подумал: «А может быть это и есть моя судьба? Может ли человек изменить свою судьбу?']

In [14]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.13 s, sys: 42.9 ms, total: 1.17 s
Wall time: 591 ms


[' – говно. И все твои книги о том, как надо жить и что делать».',
 ' – просто мудак.',
 ' – говно. И я не знаю, что хуже». Я ответил: «Это зависит от того с какой стороны посмотреть…» Он сказал мне в ответ какую-то глупость и ушел… А потом он пришел снова!',
 ' – говно. А что такое «дерьмо»? Это то, чем ты занимаешься». И я понял: да! Я действительно говнюк и ничтожество по сравнению с ним… Но ведь это не значит быть дерьмом?']

In [15]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.18 s, sys: 3.7 ms, total: 1.19 s
Wall time: 602 ms


[' – говно. И это не я говорю, а люди из твоего окружения». Я ответил: «А что такое на самом деле?',
 ' – просто говно. И ты это знаешь».',
 ' – просто свинья. И не надо меня убеждать, что это только я так думаю». В общем-то он был прав: если бы Лев Толстой жил в наше время и его спросили о том же самом… Но тогда все было по другому!',
 ' – просто говно. И я тебя не понимаю». Я промолчал, но про себя подумал: «А ведь он прав! Если бы ты был Лев Толстой на самом деле…» Но вслух ничего говорить было нельзя… А что можно?']